# 🔍 Markov Model + Diagnostic DEL Tăng Liên Tục

Notebook hoàn chỉnh bao gồm:
1. Load data
2. Build transition matrices  
3. Calibration (K values)
4. Forecast
5. **DIAGNOSTIC** - Tại sao DEL tăng sau MOB 24?
6. Giải pháp

---

## 1️⃣ SETUP & IMPORT

In [ ]:
# Setup
import sys
from pathlib import Path
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from datetime import datetime

from src.config import CFG, BUCKETS_CANON, BUCKETS_30P, BUCKETS_90P
from src.config import parse_date_column, create_segment_columns, SEGMENT_COLS
from src.data_loader import load_data
from src.rollrate.transition import compute_transition_by_mob
from src.rollrate.lifecycle import get_actual_all_vintages_amount
from src.rollrate.calibration_kmob import (
    fit_k_raw, smooth_k, fit_alpha,
    forecast_all_vintages_partial_step,
)

print("✅ Import thành công")

## 2️⃣ LOAD DATA

In [ ]:
# ========== CẤU HÌNH ==========
DATA_PATH = 'C:/Users/User/Projection_PB/Projection_pb/ETB_Parquet_YYYYMM'
MAX_MOB = 36  # Forecast đến MOB 36
# ==============================

df_raw = load_data(DATA_PATH)
df_raw['DISBURSAL_DATE'] = parse_date_column(df_raw['DISBURSAL_DATE'])
df_raw = create_segment_columns(df_raw)

print(f"📊 Data: {len(df_raw):,} rows | {df_raw[CFG['loan']].nunique():,} loans")
print(f"   Products: {df_raw['PRODUCT_TYPE'].unique().tolist()}")

## 3️⃣ BUILD TRANSITION MATRICES

In [ ]:
print("🔨 Building transition matrices...")
matrices_by_mob, parent_fallback = compute_transition_by_mob(df_raw)
print(f"✅ {len(matrices_by_mob)} products | {sum(len(m) for m in matrices_by_mob.values())} matrices")

## 4️⃣ CALIBRATION

In [ ]:
print("🔨 Calibrating k and alpha...")

# Actual results
actual_results = get_actual_all_vintages_amount(df_raw)

# DISB_TOTAL map
loan_disb = df_raw.groupby(["PRODUCT_TYPE", "RISK_SCORE", CFG["orig_date"], CFG["loan"]])[CFG["disb"]].first()
disb_total_by_vintage = loan_disb.groupby(level=[0, 1, 2]).sum().to_dict()

# Fit k_raw
k_raw_by_mob, weight_by_mob, _ = fit_k_raw(
    actual_results=actual_results,
    matrices_by_mob=matrices_by_mob,
    parent_fallback=parent_fallback,
    states=BUCKETS_CANON,
    s30_states=BUCKETS_30P,
    include_co=True,
    denom_mode="disb",
    disb_total_by_vintage=disb_total_by_vintage,
    weight_mode="equal",
    method="wls_reg",
    lambda_k=1e-4,
    k_prior=0.0,
    min_obs=5,
    fallback_k=1.0,
    fallback_weight=0.0,
    return_detail=True,
)

print(f"   K values: {len(k_raw_by_mob)} MOBs")

# Smooth k
mob_min = min(k_raw_by_mob.keys()) if k_raw_by_mob else 0
mob_max = max(k_raw_by_mob.keys()) if k_raw_by_mob else 0
k_smooth_by_mob, _, _ = smooth_k(k_raw_by_mob, weight_by_mob, mob_min, mob_max)

# Fit alpha
alpha, k_final_by_mob, _ = fit_alpha(
    actual_results=actual_results,
    matrices_by_mob=matrices_by_mob,
    parent_fallback=parent_fallback,
    states=BUCKETS_CANON,
    s30_states=BUCKETS_30P,
    k_smooth_by_mob=k_smooth_by_mob,
    mob_target=min(MAX_MOB, mob_max) if mob_max else MAX_MOB,
    include_co=True,
)

print(f"   Alpha: {alpha:.4f}")
print(f"   K_final: {len(k_final_by_mob)} MOBs")

## 5️⃣ FORECAST

In [ ]:
# Forecast với k_final
forecast_results = forecast_all_vintages_partial_step(
    actual_results=actual_results,
    matrices_by_mob=matrices_by_mob,
    parent_fallback=parent_fallback,
    max_mob=MAX_MOB,
    k_by_mob=k_final_by_mob,
    states=BUCKETS_CANON,
)

print(f"✅ Forecast: {len(forecast_results)} cohorts")

---

# 🔍 DIAGNOSTIC: TẠI SAO DEL TĂNG SAU MOB 24?

---

## 6️⃣ DIAGNOSTIC 1: Kiểm Tra K Values

In [ ]:
print("="*80)
print("1️⃣ KIỂM TRA K VALUES")
print("="*80)

print("\n   MOB  |  K value  |  Status")
print("   -----|-----------|----------")

k_issues = []
for mob in range(20, 37):
    k = k_final_by_mob.get(mob, 1.0)
    
    if k > 0.9:
        status = "❌ Rất cao"
        if mob >= 25:
            k_issues.append(f"MOB {mob}: k={k:.3f}")
    elif k > 0.7:
        status = "⚠️ Cao"
    elif k > 0.5:
        status = "✅ Trung bình"
    else:
        status = "✅ Thấp"
    
    print(f"   {mob:4d} | {k:9.3f} | {status}")

print("\n" + "-"*80)

if k_issues:
    print(f"\n❌ PHÁT HIỆN VẤN ĐỀ: {len(k_issues)} MOBs có K quá cao")
    for issue in k_issues:
        print(f"   - {issue}")
    print(f"\n💡 Giải thích:")
    print(f"   - K cao → Model tin Markov quá nhiều")
    print(f"   - Markov gây movement → DEL tăng")
    print(f"   - Cần giảm K xuống ~0.3 cho MOB 25+")
else:
    print(f"\n✅ K values hợp lý (không quá cao)")

## 7️⃣ DIAGNOSTIC 2: Kiểm Tra Fallback Usage

In [ ]:
print("\n" + "="*80)
print("2️⃣ KIỂM TRA % COHORTS DÙNG FALLBACK Ở MOB 24")
print("="*80)

total_cohorts = 0
fallback_cohorts = 0
fallback_details = []
fallback_pct = 0.0

for prod_str in matrices_by_mob.keys():
    if 24 in matrices_by_mob[prod_str]:
        for score_str in matrices_by_mob[prod_str][24].keys():
            total_cohorts += 1
            is_fallback = matrices_by_mob[prod_str][24][score_str].get("is_fallback", False)
            if is_fallback:
                fallback_cohorts += 1
                reason = matrices_by_mob[prod_str][24][score_str].get("reason", "unknown")
                fallback_details.append((prod_str, score_str, reason))

print(f"\n   Tổng cohorts: {total_cohorts}")
print(f"   Cohorts dùng fallback ở MOB 24: {fallback_cohorts}")

if total_cohorts > 0:
    fallback_pct = fallback_cohorts / total_cohorts * 100
    print(f"   Tỷ lệ: {fallback_pct:.1f}%")
    
    print("\n" + "-"*80)
    
    if fallback_pct > 30:
        print(f"\n❌ PHÁT HIỆN VẤN ĐỀ: Quá nhiều cohorts dùng fallback!")
        print(f"\n💡 Giải thích:")
        print(f"   - Các cohorts này dùng parent fallback (có movement cao)")
        print(f"   - Parent fallback tổng hợp MOB 1-24 (MOB sớm có rates cao)")
        print(f"   - Gây DEL tăng ở MOB 25+")
        
        print(f"\n   Chi tiết cohorts dùng fallback (top 10):")
        for prod, score, reason in fallback_details[:10]:
            print(f"      - {prod}/{score}: {reason}")
        if len(fallback_details) > 10:
            print(f"      ... và {len(fallback_details)-10} cohorts khác")
    elif fallback_pct > 10:
        print(f"\n⚠️ Có một số cohorts dùng fallback ({fallback_pct:.1f}%)")
    else:
        print(f"\n✅ Ít cohorts dùng fallback ({fallback_pct:.1f}%)")
else:
    print("\n⚠️ Không có cohorts nào ở MOB 24")

## 8️⃣ DIAGNOSTIC 3: So Sánh P_24 vs Parent Fallback

In [ ]:
print("\n" + "="*80)
print("3️⃣ SO SÁNH P_24 vs PARENT FALLBACK")
print("="*80)

# Tìm 1 cohort để test (không dùng fallback)
test_prod = None
test_score = None

for prod_str in matrices_by_mob.keys():
    if 24 in matrices_by_mob[prod_str]:
        for score_str in matrices_by_mob[prod_str][24].keys():
            if not matrices_by_mob[prod_str][24][score_str].get("is_fallback", False):
                test_prod = prod_str
                test_score = score_str
                break
    if test_prod:
        break

if test_prod and test_score:
    try:
        P_24 = matrices_by_mob[test_prod][24][test_score]["P"]
        key_parent = (test_prod, test_score)
        
        if key_parent in parent_fallback:
            P_parent = parent_fallback[key_parent]
            
            print(f"\n   Test cohort: {test_prod}/{test_score}")
            
            # So sánh DPD0 → DEL30+
            if "DPD0" in P_24.index and "DPD0" in P_parent.index:
                del30_states = ["DPD30+", "DPD60+", "DPD90+", "DPD120+", "DPD180+", "WRITEOFF"]
                
                p24_to_del30 = sum(P_24.loc["DPD0", s] for s in del30_states if s in P_24.columns)
                parent_to_del30 = sum(P_parent.loc["DPD0", s] for s in del30_states if s in P_parent.columns)
                
                print(f"\n   DPD0 → DEL30+ comparison:")
                print(f"   P_24:    {p24_to_del30:.4f} ({p24_to_del30*100:.2f}%)")
                print(f"   Parent:  {parent_to_del30:.4f} ({parent_to_del30*100:.2f}%)")
                print(f"   Diff:    {parent_to_del30 - p24_to_del30:+.4f} ({(parent_to_del30 - p24_to_del30)*100:+.2f}%)")
                
                print("\n" + "-"*80)
                
                if parent_to_del30 > p24_to_del30 * 1.5:
                    print(f"\n✅ XÁC NHẬN: Parent fallback có movement cao hơn P_24 nhiều")
                    print(f"   - P_24 ổn định hơn (portfolio đã mature ở MOB 24)")
                    print(f"   - Parent fallback có movement cao (tổng hợp MOB sớm)")
                elif parent_to_del30 > p24_to_del30:
                    print(f"\n✅ Parent fallback hơi cao hơn P_24")
                else:
                    print(f"\n⚠️ P_24 cao hơn hoặc bằng parent fallback")
            else:
                print("\n⚠️ Không tìm thấy DPD0 trong matrices")
        else:
            print(f"\n⚠️ Không tìm thấy parent fallback cho {test_prod}/{test_score}")
    except Exception as e:
        print(f"\n⚠️ Lỗi khi so sánh: {e}")
else:
    print("\n⚠️ Không tìm thấy cohort nào để test")

## 9️⃣ DIAGNOSTIC 4: Phân Tích Cohorts

In [ ]:
print("\n" + "="*80)
print("4️⃣ PHÂN TÍCH TỪNG COHORT")
print("="*80)

cohort_count = 0
increasing_cohorts = []
flat_cohorts = []

# Test 10 cohorts đầu
try:
    for cohort_key in list(forecast_results.keys())[:10]:
        forecast = forecast_results[cohort_key]
        disb_total = disb_total_by_vintage.get(cohort_key, 1.0)
        
        # Tính DEL30+ at MOB 24 and 30
        if 24 in forecast and 30 in forecast:
            try:
                del30_24 = forecast[24][BUCKETS_30P].sum() / disb_total
                del30_30 = forecast[30][BUCKETS_30P].sum() / disb_total
                slope = (del30_30 - del30_24) / 6
                
                cohort_count += 1
                
                if abs(slope) > 0.001:
                    increasing_cohorts.append((cohort_key, slope))
                else:
                    flat_cohorts.append((cohort_key, slope))
            except Exception as e:
                print(f"   ⚠️ Lỗi khi phân tích cohort {cohort_key}: {e}")
                continue
    
    print(f"\n   Đã phân tích {cohort_count} cohorts:")
    print(f"   - Cohorts tăng (slope > 0.001): {len(increasing_cohorts)}")
    print(f"   - Cohorts flat (slope ≤ 0.001): {len(flat_cohorts)}")
    
    if len(increasing_cohorts) > 0:
        print(f"\n   Top cohorts tăng mạnh:")
        for cohort_key, slope in sorted(increasing_cohorts, key=lambda x: abs(x[1]), reverse=True)[:5]:
            prod, score, vintage = cohort_key
            print(f"      - {prod}/{score}/{vintage}: slope = {slope:.6f} ({slope*100:.4f}%/month)")
            
            # Kiểm tra cohort này dùng fallback không
            prod_str = str(prod)
            score_str = str(score)
            if prod_str in matrices_by_mob and 24 in matrices_by_mob[prod_str]:
                if score_str in matrices_by_mob[prod_str][24]:
                    is_fallback = matrices_by_mob[prod_str][24][score_str].get("is_fallback", False)
                    if is_fallback:
                        print(f"        → ❌ Cohort này dùng FALLBACK ở MOB 24!")
                    else:
                        print(f"        → ✅ Cohort này dùng P_24 thật")
    
    print("\n" + "-"*80)
    
    if len(increasing_cohorts) > len(flat_cohorts):
        print(f"\n❌ PHÁT HIỆN VẤN ĐỀ: Nhiều cohorts vẫn tăng sau MOB 24")
    else:
        print(f"\n✅ Đa số cohorts đã flatten sau MOB 24")
        
except Exception as e:
    print(f"\n⚠️ Lỗi khi phân tích cohorts: {e}")
    increasing_cohorts = []
    flat_cohorts = []

## 🔟 KẾT LUẬN VÀ KHUYẾN NGHỊ

In [ ]:
print("\n" + "="*80)
print("KẾT LUẬN")
print("="*80)

conclusions = []

if k_issues:
    conclusions.append("❌ K values quá cao ở MOB 25+ → Tin Markov quá nhiều")

if total_cohorts > 0 and fallback_pct > 30:
    conclusions.append("❌ Nhiều cohorts dùng parent fallback ở MOB 24 → Movement cao")

if len(increasing_cohorts) > len(flat_cohorts):
    conclusions.append("❌ Nhiều cohorts vẫn tăng sau MOB 24")

if not conclusions:
    conclusions.append("✅ Không phát hiện vấn đề rõ ràng")
    conclusions.append("   → Có thể là aggregation effect hoặc weighting")

for conclusion in conclusions:
    print(f"\n{conclusion}")

print("\n" + "="*80)
print("KHUYẾN NGHỊ")
print("="*80)

if k_issues:
    print("\n1️⃣ GIẢM K Ở MOB 25+")
    print("   → Chạy cell 'Giải pháp 1' bên dưới")

if total_cohorts > 0 and fallback_pct > 30:
    print("\n2️⃣ TĂNG MIN_OBS/MIN_EAD")
    print("   → Xem hướng dẫn trong cell 'Giải pháp 2'")

if len(increasing_cohorts) > 0:
    print("\n3️⃣ KIỂM TRA CHI TIẾT COHORTS TĂNG MẠNH")
    print("   → Xem có pattern chung không (product, score, vintage)")

print("\n" + "="*80)

---

# GIẢI PHÁP

---

## Giải Pháp 1: Cap K ở MOB 25+

**Chỉ chạy cell này nếu diagnostic cho thấy: ❌ K values quá cao**

In [ ]:
print("🔧 ÁP DỤNG GIẢI PHÁP 1: Cap K ở MOB 25+")
print("="*80)

print("\nK values TRƯỚC KHI CAP:")
for mob in range(24, 37):
    k_val = k_final_by_mob.get(mob, 1.0)
    status = "❌ Cao" if k_val > 0.9 else "✅ OK"
    print(f"  MOB {mob}: {k_val:.3f} {status}")

# Cap K
print("\n🔧 Đang cap K...")
for mob in range(25, 37):
    if mob in k_final_by_mob:
        k_final_by_mob[mob] = min(k_final_by_mob[mob], 0.3)
    else:
        k_final_by_mob[mob] = 0.3

print("\nK values SAU KHI CAP:")
for mob in range(24, 37):
    k_val = k_final_by_mob.get(mob, 1.0)
    print(f"  MOB {mob}: {k_val:.3f} ✅")

print("\n" + "="*80)
print("✅ ĐÃ CAP K!")
print("="*80)
print("\n💡 Bước tiếp theo:")
print("   1. Re-run cell 5 (Forecast) với k_final_by_mob đã được cap")
print("   2. Re-run các cells diagnostic (6-10) để verify")
print("   3. Kiểm tra lại kết quả")

## Giải Pháp 2: Tăng MIN_OBS/MIN_EAD

**Chỉ áp dụng nếu diagnostic cho thấy: ❌ Nhiều cohorts dùng fallback**

In [ ]:
print("📝 GIẢI PHÁP 2: Tăng MIN_OBS/MIN_EAD")
print("="*80)
print("\n⚠️  Giải pháp này yêu cầu sửa file src/config.py")
print("\n📝 Các bước:")
print("   1. Mở file: src/config.py")
print("   2. Tìm dòng: MIN_OBS = 100")
print("   3. Sửa thành: MIN_OBS = 200")
print("   4. Tìm dòng: MIN_EAD = 1e2")
print("   5. Sửa thành: MIN_EAD = 5e2")
print("   6. Save file")
print("   7. Restart kernel và chạy lại từ đầu")
print("\n💡 Hoặc xem chi tiết trong: HUONG_DAN_CHAY_DIAGNOSTIC.md")
print("="*80)

---

## ✅ HOÀN THÀNH!

### Tóm Tắt:

1. ✅ Load data và build transition matrices
2. ✅ Calibration (K values)
3. ✅ Forecast
4. ✅ Diagnostic - Xác định vấn đề
5. ✅ Áp dụng giải pháp
6. ✅ Re-run và verify

---